# Graph vs. Hypergraph Clustering — Reproducible Runner

This notebook runs the **fair, held-out, multi-seed** benchmark on your MITAB
datasets. Place each file at `<root>/<Dataset>/<Dataset>.txt` and set `SEARCH_ROOTS`.

It fixes the issues from the original manuscript: matched-k comparison, nested
λ/k selection on a validation split, ≥20 seeds, added baselines, randomized
controls, and statistics with effect sizes. Outputs are CSVs + PDF/PNG figures.


In [2]:
pip install -r requirements.txt

  Using cached matplotlib-3.9.4-cp39-cp39-win_amd64.whl.metadata (11 kB)
  Using cached python_igraph-1.0.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached leidenalg-0.12.0-cp38-abi3-win_amd64.whl.metadata (10 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following versions that require a different python version: 3.10.0 Requires-Python >=3.10; 3.10.0rc1 Requires-Python >=3.10; 3.10.1 Requires-Python >=3.10; 3.10.3 Requires-Python >=3.10; 3.10.5 Requires-Python >=3.10; 3.10.6 Requires-Python >=3.10; 3.10.7 Requires-Python >=3.10; 3.10.8 Requires-Python >=3.10; 3.10.9 Requires-Python >=3.10; 3.11.0 Requires-Python >=3.11; 3.11.0rc1 Requires-Python >=3.11; 3.11.0rc2 Requires-Python >=3.11
ERROR: Could not find a version that satisfies the requirement markov_clustering>=0.0.6 (from versions: 0.0.2.dev0, 0.0.3.dev0, 0.0.4.dev0, 0.0.5.dev0, 0.0.6.dev0)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for markov_clustering>=0.0.6


In [3]:
# If needed:
# !pip install numpy scipy pandas scikit-learn networkx matplotlib python-igraph leidenalg markov_clustering scikit-posthocs
import sys, os
sys.path.insert(0, os.path.abspath("."))   # so `import hgspectral` works
from hgspectral.config import Config
from hgspectral import pipeline


In [4]:
cfg = Config()
cfg.dataset_names = [
    "Cardiac", "BioCreative", "Affinomics", "Cancer", "Chromatin",
    "Coronavirus", "Cyanobacteria", "Diabetes", "Crohn's_disease",
]
cfg.search_roots = ["HG_Human1", "."]   # where <Dataset>/<Dataset>.txt live
cfg.n_seeds = 25                        # >=20 for publication
cfg.out_root = "fin_result"
cfg  # review the full config (lambda grid, split fractions, baselines, ...)


Config(dataset_names=['Cardiac', 'BioCreative', 'Affinomics', 'Cancer', 'Chromatin', 'Coronavirus', 'Cyanobacteria', 'Diabetes', "Crohn's_disease"], search_roots=['HG_Human1', '.'], top_k_taxids=3, min_complex_size=2, split_fracs=(0.6, 0.2, 0.2), stratify_by_size=True, split_seed=12345, lambda_grid=(0.0, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0), k_matched='rule', min_cluster_size=5, adaptive_criteria=('eigengap', 'silhouette', 'modularity'), kmax=15, n_seeds=25, base_seed=42, n_control_seeds=10, run_controls=True, run_louvain=True, run_leiden=True, run_mcl=True, q_threshold=0.05, min_term_size=5, out_root='fin_result')

In [5]:
cfg = Config()
cfg.dataset_names = [
    "Cardiac", "BioCreative", "Affinomics", "Cancer", "Chromatin",
    "Coronavirus", "Cyanobacteria", "Diabetes", "Crohn's_disease",
]
cfg.search_roots = ["HG_Human2", "."]   # where <Dataset>/<Dataset>.txt live
cfg.n_seeds = 25                        # >=20 for publication
cfg.out_root = "fin_result"
cfg  # review the full config (lambda grid, split fractions, baselines, ...)


Config(dataset_names=['Cardiac', 'BioCreative', 'Affinomics', 'Cancer', 'Chromatin', 'Coronavirus', 'Cyanobacteria', 'Diabetes', "Crohn's_disease"], search_roots=['HG_Human2', '.'], top_k_taxids=3, min_complex_size=2, split_fracs=(0.6, 0.2, 0.2), stratify_by_size=True, split_seed=12345, lambda_grid=(0.0, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0), k_matched='rule', min_cluster_size=5, adaptive_criteria=('eigengap', 'silhouette', 'modularity'), kmax=15, n_seeds=25, base_seed=42, n_control_seeds=10, run_controls=True, run_louvain=True, run_leiden=True, run_mcl=True, q_threshold=0.05, min_term_size=5, out_root='fin_result')

## Smoke test first (synthetic — NOT real)


In [7]:
'''
from scripts.make_synthetic_mitab import make
make("SynthA/SynthA.txt", seed=0)
smoke = Config(); smoke.dataset_names=["SynthA"]; smoke.search_roots=["."]
smoke.n_seeds=6; smoke.out_root="smoke_out"
pipeline.run_all(smoke)
'''

'\nfrom scripts.make_synthetic_mitab import make\nmake("SynthA/SynthA.txt", seed=0)\nsmoke = Config(); smoke.dataset_names=["SynthA"]; smoke.search_roots=["."]\nsmoke.n_seeds=6; smoke.out_root="smoke_out"\npipeline.run_all(smoke)\n'

## Run the real benchmark with a text progress bar

This cell runs the existing pipeline once per dataset. It deliberately uses
`from tqdm import tqdm` rather than `tqdm.auto` or `tqdm.notebook`, so it does
not require Jupyter widgets. No additional Python script is needed.


In [9]:
# Self-contained dataset-level progress tracking.
# IMPORTANT: use the standard text tqdm, not tqdm.auto/tqdm.notebook.
from __future__ import annotations

import copy
import sys
import time
import traceback
from pathlib import Path

import pandas as pd
from tqdm import tqdm


def _merge_dataset_outputs(
    out_root: str | Path,
    suffix: str,
    output_name: str,
) -> Path | None:
    """
    Combine per-dataset output CSV files into one project-level CSV.

    Repeated rows are intentionally retained because different seeds and
    repetitions are valid experimental observations.
    """
    out_root = Path(out_root)

    csv_files = sorted(
        path
        for path in out_root.rglob(f"*{suffix}")
        if path.is_file() and not path.name.startswith("ALL_")
    )

    frames: list[pd.DataFrame] = []

    for csv_path in csv_files:
        try:
            frame = pd.read_csv(csv_path)

            if "dataset" not in frame.columns:
                frame.insert(0, "dataset", csv_path.parent.name)

            frame["source_file"] = str(csv_path)
            frames.append(frame)

        except Exception as exc:
            tqdm.write(
                f"[WARNING] Could not merge {csv_path}: "
                f"{type(exc).__name__}: {exc}"
            )

    if not frames:
        return None

    merged_path = out_root / output_name
    pd.concat(frames, ignore_index=True, sort=False).to_csv(
        merged_path,
        index=False,
    )
    return merged_path


def run_all_datasets_with_progress(
    cfg,
    *,
    continue_on_error: bool = True,
    merge_outputs: bool = True,
) -> pd.DataFrame:
    """
    Run the existing research pipeline one dataset at a time.

    Parameters
    ----------
    cfg
        Existing hgspectral Config object.

    continue_on_error
        When True, record a failed dataset and continue with the next one.

    merge_outputs
        When True, reconstruct project-level ALL_*.csv files after all
        datasets have been attempted.

    Returns
    -------
    pandas.DataFrame
        Dataset-level status, timing, and error information.
    """
    datasets = list(cfg.dataset_names)

    if not datasets:
        raise ValueError(
            "cfg.dataset_names is empty. Add at least one dataset."
        )

    out_root = Path(cfg.out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    status_path = out_root / "dataset_progress.csv"
    records: list[dict] = []

    # file=sys.stdout and standard tqdm force a non-widget text progress bar.
    progress = tqdm(
        datasets,
        total=len(datasets),
        desc="Datasets",
        unit="dataset",
        dynamic_ncols=True,
        leave=True,
        file=sys.stdout,
    )

    for index, dataset_name in enumerate(progress, start=1):
        progress.set_description_str(
            f"Dataset {index}/{len(datasets)}: {dataset_name}"
        )
        progress.set_postfix_str("running", refresh=True)

        started = time.perf_counter()
        status = "completed"
        error_type = ""
        error_message = ""

        # Isolate each run so the original cfg is not modified.
        run_cfg = copy.deepcopy(cfg)
        run_cfg.dataset_names = [dataset_name]

        try:
            pipeline.run_all(run_cfg)

        except KeyboardInterrupt:
            progress.set_postfix_str("interrupted", refresh=True)
            raise

        except Exception as exc:
            status = "failed"
            error_type = type(exc).__name__
            error_message = str(exc)

            tqdm.write(
                f"\n[ERROR] {dataset_name}: "
                f"{error_type}: {error_message}",
                file=sys.stdout,
            )
            traceback.print_exc()

            if not continue_on_error:
                raise

        finally:
            elapsed_seconds = time.perf_counter() - started

            records.append(
                {
                    "dataset": dataset_name,
                    "status": status,
                    "elapsed_seconds": round(elapsed_seconds, 3),
                    "elapsed_minutes": round(elapsed_seconds / 60, 3),
                    "error_type": error_type,
                    "error_message": error_message,
                }
            )

            # Crash-safe checkpoint after every dataset.
            pd.DataFrame(records).to_csv(status_path, index=False)

            progress.set_postfix_str(
                f"{status}; {elapsed_seconds / 60:.2f} min",
                refresh=True,
            )

            tqdm.write(
                f"[{status.upper()}] {dataset_name} "
                f"({elapsed_seconds / 60:.2f} minutes)",
                file=sys.stdout,
            )

    progress.close()
    status_df = pd.DataFrame(records)

    if merge_outputs:
        merged_outputs = {
            "test metrics": _merge_dataset_outputs(
                out_root,
                "_test_metrics.csv",
                "ALL_test_metrics.csv",
            ),
            "controls": _merge_dataset_outputs(
                out_root,
                "_controls.csv",
                "ALL_controls.csv",
            ),
            "GO summaries": _merge_dataset_outputs(
                out_root,
                "_go_enrichment_summary.csv",
                "ALL_go_enrichment_summary.csv",
            ),
            "manifests": _merge_dataset_outputs(
                out_root,
                "_manifest.csv",
                "ALL_manifest.csv",
            ),
        }

        for label, merged_path in merged_outputs.items():
            if merged_path is not None:
                print(f"Merged {label}: {merged_path}")

    completed = int((status_df["status"] == "completed").sum())
    failed = int((status_df["status"] == "failed").sum())

    print(
        f"\nFinished: {completed}/{len(datasets)} completed; "
        f"{failed} failed."
    )
    print(f"Progress record: {status_path}")

    return status_df


# This replaces the original: pipeline.run_all(cfg)
status_df = run_all_datasets_with_progress(
    cfg,
    continue_on_error=True,
    merge_outputs=True,
)

status_df


Dataset 1/9: Cardiac:   0%|                                                        | 0/9 [00:00<?, ?dataset/s, running][Cardiac] n_train=1152 k_matched=15 selected(lambda*=0.0, k*=15) val_bestF1=0.285
[DONE] 1 datasets in 72.9s -> fin_result
[COMPLETED] Cardiac (1.22 minutes)                                                                                     
Dataset 2/9: BioCreative:  11%|████▉                                       | 1/9 [01:12<09:43, 72.94s/dataset, running][BioCreative] n_train=327 k_matched=15 selected(lambda*=0.001, k*=15) val_bestF1=0.201
[DONE] 1 datasets in 16.1s -> fin_result
[COMPLETED] BioCreative (0.27 minutes)                                                                                 
Dataset 3/9: Affinomics:  22%|██████████                                   | 2/9 [01:29<04:36, 39.49s/dataset, running][Affinomics] n_train=326 k_matched=15 selected(lambda*=0.0, k*=15) val_bestF1=0.260
[DONE] 1 datasets in 16.5s -> fin_result
[COMPLETED] Affinomics (0.2

,dataset,status,elapsed_seconds,elapsed_minutes,error_type,error_message
0,Cardiac,completed,72.931,1.216,,
1,BioCreative,completed,16.056,0.268,,
2,Affinomics,completed,16.541,0.276,,
3,Cancer,completed,1209.526,20.159,,
4,Chromatin,completed,296.955,4.949,,
5,Coronavirus,completed,732.352,12.206,,
6,Cyanobacteria,completed,19.987,0.333,,
7,Diabetes,completed,447.778,7.463,,
8,Crohn's_disease,completed,8.916,0.149,,


## Run the real benchmark
Each dataset writes `*_test_metrics.csv`, `*_controls.csv`,
`*_go_enrichment_summary.csv`, `*_manifest.csv` under `fin_result/<Dataset>/`.

## Aggregate, run statistics, render figures

In [12]:
import subprocess, sys
subprocess.run([sys.executable, "analyze_results.py",
                "--results", "fin_result/ALL_test_metrics.csv"], check=False)
import pandas as pd
pd.read_csv("fin_result/analysis/summary_by_method.csv").head(20)


,dataset,method,k_mode,n_seeds,pairwise_f1_mean,pairwise_f1_std,bestmatch_f1_sym_mean,bestmatch_f1_sym_std,b3_f1_mean,b3_f1_std,NMI_mean,NMI_std,ARI_mean,ARI_std,hyperedge_recovery_f1_mean,hyperedge_recovery_f1_std
0,Affinomics,graph_spectral,matched,25,0.038960,0.001181,0.181514,0.005866,0.038788,1.982926e-03,0.732886,7.228743e-03,0.172726,0.005718,0.166199,0.008277
1,Affinomics,graph_spectral_adaptive,adaptive,25,0.018748,0.000628,0.099027,0.001048,0.019589,5.860639e-04,0.584674,5.436424e-03,0.090018,0.004522,0.075226,0.002089
2,Affinomics,hyper_penalized,matched,25,0.040798,0.004076,0.203218,0.006679,0.052176,1.795823e-03,0.764345,9.156031e-03,0.206743,0.009503,0.189101,0.008674
3,Affinomics,hyper_penalized_adaptive,adaptive,25,0.013198,0.000646,0.076344,0.002868,0.015078,7.090588e-04,0.483545,1.478389e-02,0.054480,0.005440,0.056141,0.001652
4,Affinomics,hyper_zhou_lam0,matched,25,0.040798,0.004076,0.203218,0.006679,0.052176,1.795823e-03,0.764345,9.156031e-03,0.206743,0.009503,0.189101,0.008674
5,Affinomics,louvain,native,5,0.083696,0.001884,0.320270,0.005758,0.101806,1.996157e-03,0.843931,5.558155e-03,0.295062,0.020164,0.389342,0.008419
6,Affinomics,mcl,native,1,0.149644,0.000000,0.410418,0.000000,0.134989,0.000000e+00,0.896403,0.000000e+00,0.407829,0.000000,0.544143,0.000000
7,BioCreative,graph_spectral,matched,25,0.039476,0.001179,0.192061,0.005065,0.042460,2.007911e-03,0.765243,8.959777e-03,0.194093,0.018260,0.173221,0.006570
8,BioCreative,graph_spectral_adaptive,adaptive,25,0.022694,0.001264,0.129357,0.002555,0.026312,1.071396e-03,0.675810,5.984865e-03,0.123979,0.007248,0.107395,0.003310
9,BioCreative,hyper_penalized,matched,25,0.033745,0.001852,0.193979,0.004852,0.046654,1.713573e-03,0.769059,7.926063e-03,0.201441,0.011813,0.174267,0.005529


In [13]:
# Paired graph-vs-hypergraph tests (matched-k, held-out test), with effect sizes + CIs
pd.read_csv("fin_result/analysis/paired_graph_vs_hyper.csv")


,dataset,metric,hyper_mean,graph_mean,n,statistic,p_value,rank_biserial,median_diff,ci_mean_diff,ci_lo,ci_hi
0,Affinomics,bestmatch_f1_sym,0.203218,0.181514,25,0.0,5.960464e-08,1.000000,0.019139,0.021704,0.018452,0.025088
1,BioCreative,bestmatch_f1_sym,0.193979,0.192061,25,115.0,2.099392e-01,0.292308,0.003960,0.001918,-0.001561,0.005178
2,Cancer,bestmatch_f1_sym,0.048544,0.048800,25,131.0,4.107652e-01,-0.193846,0.000015,-0.000256,-0.000823,0.000314
3,Cardiac,bestmatch_f1_sym,0.112376,0.114887,25,53.0,2.255261e-03,-0.673846,-0.002452,-0.002511,-0.003804,-0.001211
4,Chromatin,bestmatch_f1_sym,0.243785,0.206767,25,0.0,5.960464e-08,1.000000,0.036994,0.037018,0.035196,0.038432
5,Coronavirus,bestmatch_f1_sym,0.240591,0.241848,25,136.0,4.907860e-01,-0.163077,-0.000337,-0.001256,-0.003981,0.001559
6,Crohn's_disease,bestmatch_f1_sym,0.543610,0.531969,25,31.0,1.398921e-04,0.809231,0.003855,0.011641,0.005772,0.017485
7,Cyanobacteria,bestmatch_f1_sym,0.493771,0.471219,25,0.0,5.960464e-08,1.000000,0.024790,0.022552,0.020321,0.024260
8,Diabetes,bestmatch_f1_sym,0.233903,0.240634,25,64.0,6.725550e-03,-0.606154,-0.006164,-0.006730,-0.011649,-0.001436


In [14]:
print(open("fin_result/analysis/friedman_nemenyi.txt").read())


Friedman p = 0.3281495877705818

Mean ranks (1=best):
method
graph_spectral     1.888889
hyper_zhou_lam0    2.388889
hyper_penalized    1.722222
dtype: float64



### Notes / knobs
- `cfg.lambda_grid` — λ sweep (includes 0 for the unregularized ablation).
- `cfg.stratify_by_size` — size-stratified hyperedge split (default True).
- `cfg.k_matched` — `"rule"` (n_train // min_cluster_size, capped) or an int.
- For weighting/min-size/species ablations, copy `cfg`, change one field, rerun
  into a different `out_root`, then compare the summary CSVs.
